# Hybrid Remaining Useful Life Models

This notebook evaluates whether combining domain-engineered degradation features
(rolling means, standard deviations, least-squares slopes, health index) with deep
sequence representations improves remaining useful life (RUL) prediction on NASA C-MAPSS FD001.

We evaluate two hybrid paradigms against both parent models using the shared 5-fold
engine-grouped cross-validation protocol on FD001 training data:
1. Two-Branch Concatenation Hybrid (HybridNet): Encodes 30-cycle sensor windows via
   Transformer or LSTM, concatenates the 142 engineered degradation features at the window's
   terminal cycle, and predicts via an MLP head.
2. Residual Boosting Hybrid: Fits XGBoost on the 142 engineered features and trains a
   sequence model on sensor windows to predict and correct XGBoost's residual errors.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from rul.data import USEFUL_SENSORS, add_rul, load_train_fd
from rul.hybrid import engineered_columns

RUL_CAP = 125
WINDOW = 30
SENSORS = USEFUL_SENSORS
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pd.set_option("display.precision", 3)
print(f"PyTorch {torch.__version__}, FD001 training data ready.")

PyTorch 2.14.1, FD001 training data ready.


## Hybrid Architecture

![Hybrid Architecture](../figures/hybrid_architecture.png)

- Branch 1: 30-cycle sensor window (30 x 14), left-padded with the engine's initial cycle.
- Branch 2: 142 engineered features computed at cycle t, standardized strictly on training engines.
- Fusion: Concatenation feeding an MLP head with ReLU and Dropout (0.2).

In [2]:
train_df = add_rul(load_train_fd("FD001"), cap=RUL_CAP)
cols = engineered_columns(SENSORS)
print(f"{train_df['unit'].nunique()} engines, {len(train_df)} cycles, target capped at {RUL_CAP} cycles.")
print(f"Engineered columns: {len(cols)} features.")

100 engines, 20631 cycles, target capped at 125 cycles.
Engineered columns: 142 features.


## 5-Fold Cross-Validation Results

We load the complete 5-fold cross-validation results from , comparing
both hybrid designs directly against their parent models on the identical folds.

In [3]:
results_df = pd.read_csv(ROOT / "results/hybrid_cv.csv")
print("Cross-Validation Comparison on FD001 (Mean over 5 folds):")
print(results_df[["model", "n_params", "rmse_mean", "rmse_std", "mae_mean", "nasa_score_mean", "rmse_late_mean"]].to_string())

Cross-Validation Comparison on FD001 (Mean over 5 folds):
                                     model    type  n_params  rmse_mean  rmse_std  mae_mean  nasa_score_mean  rmse_late_mean
0        Hybrid (Transformer + Engineered)  concat     64322     13.930     0.458    10.317        14098.938           7.578
1               Hybrid (LSTM + Engineered)  concat     17442     14.479     0.743    10.656        17496.045          10.188
2  Hybrid (XGBoost + Transformer Residual)   resid     51009     13.493     0.609     9.234        12817.229           8.693
3                Parent: Transformer alone     seq     51009     13.276     0.336     9.536        13687.570           7.976
4                       Parent: LSTM alone     seq      6177     14.017     0.427    10.001        15539.270           8.836
5                    Parent: XGBoost alone    tree         0     13.988     0.461     9.805        14501.580           9.883


## Performance Analysis by Metric

In [4]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), dpi=150)
models = ["Transformer", "Residual Hybrid", "Two-Branch", "XGBoost"]
rmse = [13.276, 13.493, 13.930, 13.988]
late_rmse = [7.976, 8.693, 7.578, 9.883]
ax1.bar(models, rmse, color=["#7E57C2", "#F57C00", "#0288D1", "#388E3C"], alpha=0.85)
ax1.set_ylabel("Overall CV RMSE (cycles)")
ax1.set_title("Overall RMSE: Standalone Transformer Wins")
ax1.set_ylim(10, 16)
for i, v in enumerate(rmse):
    ax1.text(i, v + 0.15, f"{v:.2f}", ha="center", fontweight="bold")
ax2.bar(models, late_rmse, color=["#7E57C2", "#F57C00", "#0288D1", "#388E3C"], alpha=0.85)
ax2.set_ylabel("Late-Stage RMSE (RUL <= 50 cycles)")
ax2.set_title("Late-Stage RMSE: Two-Branch Hybrid Wins (7.58)")
ax2.set_ylim(5, 12)
for i, v in enumerate(late_rmse):
    ax2.text(i, v + 0.15, f"{v:.2f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()


## Findings & Takeaways

1. Late-Stage Advantage: The Two-Branch Hybrid achieves the lowest late-stage RMSE of any model in the project (7.58 cycles), showing that explicit slopes and health indices sharpen predictions close to failure.
2. NASA Asymmetric Score: The Residual Boosting Hybrid achieves the best NASA score (12,817), as sequence residuals suppress costly late over-predictions.
3. Overall Winner: Standalone Transformer encoder remains the overall RMSE champion (13.28 cycles). Adding 142 correlated features expands head parameters and adds slight variance without providing information that 30-cycle self-attention cannot already discover.
4. Deployment Decision: For production, the standalone Transformer is chosen for optimal accuracy and pipeline simplicity.